In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))
import inventory as I
import manifest as M

In [ ]:
rbh_dir = Path("/home/jormond/Documents/DS217/users/jormond/RBH")
smh_dir = Path("/home/jormond/Documents/DS217/users/jormond/SMH")
(open_picu_dir/"derived").mkdir(exist_ok=True)
(open_picu_dir/"reports").mkdir(exist_ok=True)

rbh_m, rbh_w = M.rbh_manifest(rbh_dir)
smh_m, smh_w = M.smh_manifest(smh_dir)

In [ ]:
M.print_waterfall(rbh_w, "RBH")
M.print_waterfall(smh_w, "SMH")

In [ ]:
rbh_years = M.describe(rbh_m, "PATIENT_RK")
smh_years = M.describe(smh_m, "UniquePtID")
pd.concat({"RBH": rbh_years["stays"], "SMH": smh_years["stays"]}, axis=1)

In [ ]:
rbh_m.to_csv(open_picu_dir/"derived"/"RBH_manifest.csv", index=False)
smh_m.to_csv(open_picu_dir/"derived"/"SMH_manifest.csv", index=False)

### 2013 and beyond stays not in picanet

In [ ]:
dem = pd.read_csv(rbh_dir/"ai_picu_demographics.csv", usecols=["studyID", "PICUadm_dttm"]).drop_duplicates("studyID")
dem["stay_key"] = M.rbh_stay_key(dem["studyID"])
dem["year"] = pd.to_datetime(dem["PICUadm_dttm"], format=M.RBH_TIME_FMT).dt.year
dropped = dem[(dem["year"] >= 2013) & ~dem["stay_key"].isin(set(rbh_m["stay_key"]))].drop_duplicates("stay_key")
print(dropped["year"].value_counts().sort_index().to_string())